# Laboratorio 2A — Carga de datos y manejo de DataFrames

Antes de auditar un conjunto de datos hay que poder abrirlo, describirlo y recortarlo con
soltura. Este laboratorio fija ese piso: **cargar un archivo, entender qué trajo, seleccionar,
filtrar, agrupar y resumir**. No hay algoritmos acá; hay manejo de la herramienta con la que se
trabaja el resto del curso.

El conjunto de datos es real: **Heart Failure Prediction**, 918 pacientes y 12 variables
clínicas, publicado sin credencialización. Se carga directamente desde su URL, así que el
notebook abre en Google Colab sin subir nada.

Las celdas que contienen `raise NotImplementedError` deben completarse.

## 0. Preparación

`pandas` y `matplotlib` vienen instalados en Colab. La única dependencia externa es la conexión
para descargar el archivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Cargar un CSV

`pd.read_csv` acepta una ruta local o una URL. En un archivo local sería
`pd.read_csv("datos/heart.csv")`; acá se lee directo del repositorio público.

In [ ]:
URL = "https://raw.githubusercontent.com/gustavovazquez/datasets/main/heart.csv"

df = pd.read_csv(URL)
df.shape

Los argumentos de `read_csv` que más se usan en la práctica:

| Argumento | Para qué |
|---|---|
| `sep` | Separador, cuando no es la coma (`sep=";"` es habitual en archivos europeos) |
| `decimal` | Separador decimal (`decimal=","`) |
| `na_values` | Qué cadenas se leen como faltante (`na_values=["", "NA", "sin dato"]`) |
| `dtype` | Forzar el tipo de una columna (`dtype={"id": str}`, para no perder ceros a la izquierda) |
| `usecols` | Leer solo algunas columnas, cuando el archivo es grande |
| `nrows` | Leer las primeras filas, para inspeccionar antes de cargar todo |

**Un identificador se lee siempre como texto.** Si se lee como entero, `00734` se convierte en
`734` y el identificador queda destruido.

## 2. Primer vistazo

Las cuatro operaciones que se hacen siempre, en este orden: forma, primeras filas, tipos y
resumen numérico.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

`describe()` solo resume las columnas numéricas. Para las categóricas hay que pedirlo
explícitamente.

In [ ]:
df.describe(include="object")

### Para analizar

`RestingBP` (presión en reposo) y `Cholesterol` tienen mínimo 0. Un colesterol de 0 mg/dl no es
una medición **posible**: es un código que la institución usó para «no medido».

Responder a partir de la salida de `describe()`:

1. ¿Cuántos registros tienen `Cholesterol == 0`? ¿Qué porcentaje del total representan?
2. Si esos ceros se dejan como están, ¿en qué dirección se desplaza la media de `Cholesterol`?

In [ ]:
# TODO: contar los registros con Cholesterol == 0 y su porcentaje sobre el total.
raise NotImplementedError()

**Respuesta:**

1.
2.

## 3. Selección de columnas y filas

Tres formas que conviene no mezclar:

| Sintaxis | Qué devuelve |
|---|---|
| `df["Age"]` | Una **Series** (una columna) |
| `df[["Age", "Sex"]]` | Un **DataFrame** (lista de columnas) |
| `df.loc[filas, columnas]` | Selección **por etiqueta** |
| `df.iloc[filas, columnas]` | Selección **por posición** |

In [ ]:
print(type(df["Age"]))
print(type(df[["Age"]]))

df.loc[0:4, ["Age", "Sex", "Cholesterol", "HeartDisease"]]

**Cuidado con `loc` e `iloc`:** `df.loc[0:4]` incluye la fila 4; `df.iloc[0:4]` no. `loc`
trabaja con etiquetas y el rango es cerrado; `iloc` trabaja con posiciones y sigue la convención
de Python.

In [ ]:
print("loc[0:4] devuelve", len(df.loc[0:4]), "filas")
print("iloc[0:4] devuelve", len(df.iloc[0:4]), "filas")

## 4. Filtrado por condición

Una condición sobre una Series devuelve una máscara booleana, y esa máscara indexa el
DataFrame. Las condiciones se combinan con `&` (y), `|` (o) y `~` (no), **cada una entre
paréntesis**.

In [ ]:
mascara = (df["Age"] > 60) & (df["HeartDisease"] == 1)
print(f"{mascara.sum()} pacientes mayores de 60 años con enfermedad")

df.loc[mascara, ["Age", "Sex", "MaxHR", "Oldpeak"]].head()

Corresponde implementar una función de filtrado. Es un envoltorio delgado, pero fuerza a
manejar la máscara con explícitud.

In [ ]:
def filtrar(df: pd.DataFrame, columna: str, minimo: float, maximo: float) -> pd.DataFrame:
    """Devuelve las filas cuyo valor en `columna` cae en el intervalo cerrado [minimo, maximo].

    Parámetros
    ----------
    df : DataFrame de entrada.
    columna : nombre de una columna numérica.
    minimo, maximo : extremos del intervalo, incluidos.

    Devuelve
    --------
    DataFrame con las filas seleccionadas y todas las columnas originales.
    Los faltantes de `columna` no se seleccionan.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
adultos = filtrar(df, "Age", 40, 50)
assert adultos["Age"].between(40, 50).all(), "quedaron filas fuera del intervalo"
assert len(adultos) == int(df["Age"].between(40, 50).sum()), "faltan o sobran filas"
print(f"OK — {len(adultos)} pacientes entre 40 y 50 años")

## 5. Columnas derivadas

Una columna nueva se crea asignando sobre el DataFrame. Para clasificar por tramos,
`pd.cut` es la herramienta directa.

In [ ]:
df["riesgo_hr"] = np.where(df["MaxHR"] < 120, "bajo", "normal")

df["grupo_edad"] = pd.cut(
    df["Age"],
    bins=[0, 40, 55, 70, 120],
    labels=["hasta 40", "41-55", "56-70", "más de 70"],
)

df[["Age", "grupo_edad", "MaxHR", "riesgo_hr"]].head()

**`SettingWithCopyWarning`.** Asignar sobre un recorte (`sub = df[df.Age > 60]` y después
`sub["x"] = ...`) avisa que no se sabe si se modifica el original o una copia. La forma correcta
es pedir la copia explícitamente: `sub = df.loc[df.Age > 60].copy()`.

In [ ]:
sub = df.loc[df["Age"] > 60].copy()
sub["edad_relativa"] = sub["Age"] - df["Age"].mean()
sub[["Age", "edad_relativa"]].head(3)

## 6. Agrupar y resumir

`groupby` parte el DataFrame según una clave y aplica una función de resumen a cada grupo. Es
la operación que más se usa en una auditoría: casi toda pregunta interesante es «¿cómo cambia
esto **según** aquello?».

In [ ]:
df.groupby("ChestPainType")["HeartDisease"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("Sex").agg(
    n=("Age", "size"),
    edad_media=("Age", "mean"),
    colesterol_mediano=("Cholesterol", "median"),
    tasa_enfermedad=("HeartDisease", "mean"),
).round(2)

Corresponde implementar el resumen por grupo. La media y la mediana juntas anticipan la
comparación de la clase: cuando difieren mucho, hay asimetría o valores extremos.

In [ ]:
def resumen_por_grupo(df: pd.DataFrame, grupo: str, variable: str) -> pd.DataFrame:
    """Resume `variable` dentro de cada nivel de `grupo`.

    Parámetros
    ----------
    df : DataFrame de entrada.
    grupo : nombre de una columna categórica.
    variable : nombre de una columna numérica.

    Devuelve
    --------
    DataFrame indexado por los niveles de `grupo`, con las columnas
    ["n", "media", "mediana", "desvio"], ordenado de mayor a menor mediana.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
r = resumen_por_grupo(df, "ChestPainType", "MaxHR")
assert list(r.columns) == ["n", "media", "mediana", "desvio"], "columnas o su orden"
assert r["n"].sum() == len(df), "los grupos no cubren todas las filas"
assert r["mediana"].is_monotonic_decreasing, "falta ordenar por mediana descendente"
print(r.round(2))

### Para analizar

1. ¿Qué tipo de dolor de pecho concentra la mayor tasa de enfermedad?
2. En `MaxHR` por tipo de dolor, ¿la media y la mediana cuentan la misma historia en todos los
   grupos? ¿En cuál se separan más, y qué sugiere esa separación?

**Respuesta:**

1.
2.

## 7. Tablas cruzadas y conteos

Para dos categóricas, `crosstab` da la tabla de contingencia; con `normalize` da proporciones.

In [ ]:
pd.crosstab(df["ChestPainType"], df["HeartDisease"])

In [ ]:
pd.crosstab(df["ChestPainType"], df["HeartDisease"], normalize="index").round(3)

## 8. Ordenar, contar y valores únicos

In [ ]:
print(df["ST_Slope"].value_counts())
print()
print("Niveles distintos de ChestPainType:", df["ChestPainType"].nunique())
print("Cuáles:", sorted(df["ChestPainType"].unique()))
print()
df.sort_values("Cholesterol", ascending=False).head(3)[["Age", "Sex", "Cholesterol"]]

## 9. Faltantes y tipos

Este conjunto no trae `NaN`, pero sí trae faltantes **disfrazados de cero**. Es el caso más
frecuente en datos reales, y el motivo por el que `isna().sum()` nunca alcanza como única
revisión.

In [ ]:
print("Faltantes declarados por columna:")
print(df.isna().sum().to_string())

sin_codigo = df.replace({"Cholesterol": {0: np.nan}, "RestingBP": {0: np.nan}})
print()
print("Faltantes después de reconocer los códigos:")
print(sin_codigo.isna().sum().to_string())

### DECIDE

`Cholesterol == 0` aparece en 172 registros. Hay tres caminos: dejarlo, convertirlo a `NaN`, o
eliminar esas filas.

1. ¿Cuál corresponde en esta etapa del trabajo y por qué?
2. ¿Qué información se pierde con cada uno de los otros dos?

**Respuesta:**

1.
2.

## 10. Guardar el resultado

Toda transformación intermedia se guarda con un nombre que dice qué se hizo. `index=False`
evita que el índice se escriba como una columna anónima que reaparece en la próxima carga.

In [ ]:
sin_codigo.to_csv("heart_sin_codigos.csv", index=False)
control = pd.read_csv("heart_sin_codigos.csv")
print(control.shape, "— faltantes:", int(control.isna().sum().sum()))

## Cierre

- `read_csv` es donde se decide el tipo de cada columna. Un identificador leído como entero ya
  llega dañado a la auditoría.
- `describe()` es la primera herramienta de detección de errores: un mínimo de 0 en una variable
  fisiológica es un código de error, no una medición.
- `groupby` responde la pregunta que importa en una auditoría: cómo cambia una variable **según**
  otra. Es la base de lo que sigue.

El laboratorio siguiente usa exactamente estas operaciones sobre el conjunto de datos del curso.